# 대분류(6종) 카테고리 + 색상 고도화 검증 노트북

**스파이크/검증용 코드입니다. 프로덕션 코드가 아닙니다.**

`product_info_check.ipynb`(소분류 30종 + 카테고리 프롬프트 v1~v4 검증)와 별개 노트북이다 —
이번엔 **대분류(OUTER/TOP/BOTTOM/SHOES/BAG/ACCESSORY) 6종만** 분류했을 때 CLIP과 VLM이
각각 어떤지 비교하고, 색상은 지금까지 나온 개선안(프롬프트 v2/v3 + 화이트밸런스 전처리 +
팔레트 스와치 참조)을 한 자리에서 재실측한다.

## 사용법

1. §2(대분류)·§3(대분류 VLM)·§5(색상 VLM 계열)는 `RUN_VLM = True`로 바꿔야 실행된다
   (Qwen3-VL 로드 — 가볍지 않다).
2. 정답셋은 `product_info_check.ipynb`와 같은 `data/ground_truth.json`을 그대로 쓴다 —
   대분류 정답은 `taxonomy.major_of(subcategory)`로 즉석에서 계산한다(별도 라벨링 불필요).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))

import pandas as pd

from classify import classify_major_only, get_category_classifier
from color import PALETTE, build_swatch_reference
from dataset import load_ground_truth
from taxonomy import MAJOR_CATEGORIES, major_of

gt = load_ground_truth()
real_samples = [s for s in gt.samples if not s.synthetic]
print(f"정답셋: {len(gt.samples)}행(문제 {len(gt.issues)}행), 실제 사진 {len(real_samples)}장")
print(f"대분류 목록: {MAJOR_CATEGORIES}")


정답셋: 18행(문제 0행), 실제 사진 17장
대분류 목록: ('OUTER', 'TOP', 'BOTTOM', 'SHOES', 'BAG', 'ACCESSORY')


## 1. 설정 상수

`RUN_VLM`을 `True`로 바꿔야 아래 Qwen3-VL 관련 셀이 실행된다. `CATEGORY_TEMPLATE_KEY`는
CLIP 대분류 분류에 쓰는 템플릿(`taxonomy.MAJOR_PROMPT_TEMPLATES`의 키) — 소분류 검증에서
`en_clip_style`이 가장 좋았으므로 기본값으로 그대로 가져온다.

In [2]:
RUN_VLM = True  # <- 명시적으로 True로 바꿔야 Qwen3-VL 관련 셀이 실행됨
CATEGORY_BACKEND = "clip_vit_b32"  # classify.CATEGORY_BACKENDS의 키
CATEGORY_TEMPLATE_KEY = "en_clip_style"  # taxonomy.MAJOR_PROMPT_TEMPLATES의 키

print(f"CATEGORY_BACKEND={CATEGORY_BACKEND}, CATEGORY_TEMPLATE_KEY={CATEGORY_TEMPLATE_KEY}, RUN_VLM={RUN_VLM}")


CATEGORY_BACKEND=clip_vit_b32, CATEGORY_TEMPLATE_KEY=en_clip_style, RUN_VLM=True


## 2. 대분류 분류 — CLIP zero-shot

`classify.classify_major_only()` — 대분류 6종만 물어본다(소분류 30-way보다 후보가 훨씬
적어 CLIP이 더 잘할 가능성이 있다).

In [3]:
clip_classifier = get_category_classifier(CATEGORY_BACKEND)

clip_major_rows = []
for s in real_samples:
    result = classify_major_only(clip_classifier, s.image_path, CATEGORY_TEMPLATE_KEY)
    truth_major = major_of(s.subcategory)
    clip_major_rows.append(
        {
            "image_file": s.image_file,
            "truth_major": truth_major,
            "pred_major": result.predicted,
            "correct": result.predicted == truth_major,
            "score": round(result.score, 3),
        }
    )

clip_major_df = pd.DataFrame(clip_major_rows)
clip_major_acc = clip_major_df["correct"].mean()
print(f"[CLIP {CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY}] 대분류 top1 = {clip_major_acc:.2f}")
clip_major_df


c:\Users\SSAFY\miniforge3\envs\qual\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 21287.90it/s]


[CLIP clip_vit_b32/en_clip_style] 대분류 top1 = 0.35


,image_file,truth_major,pred_major,correct,score
0,denim.jpg,BOTTOM,BOTTOM,True,0.592
1,black_washed.jpg,TOP,OUTER,False,0.909
2,fix.jpg,BOTTOM,BOTTOM,True,0.518
3,levis_patch.jpg,BOTTOM,OUTER,False,0.413
4,logo_test.jpg,TOP,OUTER,False,0.825
5,scratch.jpg,BOTTOM,OUTER,False,0.490
6,small_pro.jpg,TOP,OUTER,False,0.680
7,navy_logo.png,TOP,OUTER,False,0.923
8,vintage_t.png,TOP,OUTER,False,0.712
9,black_stain.png,TOP,OUTER,False,0.730


### 2-1. 대분류 분류 — CLIP은 소분류(30-way)로 물은 뒤 대분류로 역산

방금 §2에서 CLIP에게 대분류를 직접 물으면 "OUTER" 쏠림으로 0.20까지 떨어지는 걸 봤다.
반면 소분류 30-way(`en_clip_style`)는 `product_info_check.ipynb`/`REPORT.md` §8에서
0.71~0.90이 나왔었다 — **CLIP에게는 소분류로 물어서 받고, 사용자에게는
`taxonomy.major_of()`로 역산한 대분류만 보여주면 정확도를 그대로 가져갈 수 있지
않을까**(2026-08-03, 사용자 제안)를 실측으로 확인한다.

In [5]:
from classify import classify_one_stage

clip_sub_to_major_rows = []
for s in real_samples:
    result = classify_one_stage(clip_classifier, s.image_path, CATEGORY_TEMPLATE_KEY)
    truth_major = major_of(s.subcategory)
    pred_major = major_of(result.predicted)
    clip_sub_to_major_rows.append(
        {
            "image_file": s.image_file,
            "truth_subcategory": s.subcategory,
            "pred_subcategory": result.predicted,
            "truth_major": truth_major,
            "pred_major": pred_major,
            "correct_subcategory": result.predicted == s.subcategory,
            "correct_major": pred_major == truth_major,
        }
    )

clip_sub_to_major_df = pd.DataFrame(clip_sub_to_major_rows)
clip_sub_to_major_acc = clip_sub_to_major_df["correct_major"].mean()
print(
    f"[CLIP {CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY}, 소분류->대분류 역산] "
    f"대분류 top1 = {clip_sub_to_major_acc:.2f}  (직접 대분류 질의 대비 {clip_major_acc:.2f})"
)
clip_sub_to_major_df


[CLIP clip_vit_b32/en_clip_style, 소분류->대분류 역산] 대분류 top1 = 1.00  (직접 대분류 질의 대비 0.35)


,image_file,truth_subcategory,pred_subcategory,truth_major,pred_major,correct_subcategory,correct_major
0,denim.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
1,black_washed.jpg,T_SHIRT,T_SHIRT,TOP,TOP,True,True
2,fix.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
3,levis_patch.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
4,logo_test.jpg,T_SHIRT,SHIRT,TOP,TOP,False,True
5,scratch.jpg,DENIM,DENIM,BOTTOM,BOTTOM,True,True
6,small_pro.jpg,T_SHIRT,SHIRT,TOP,TOP,False,True
7,navy_logo.png,T_SHIRT,T_SHIRT,TOP,TOP,True,True
8,vintage_t.png,T_SHIRT,T_SHIRT,TOP,TOP,True,True
9,black_stain.png,T_SHIRT,SHIRT,TOP,TOP,False,True


## 3. 대분류 분류 — VLM(Qwen3-VL)

`vlm_control.query_major_category()` — 대분류 6종 프롬프트에 OUTER/TOP, BAG/ACCESSORY
구분 기준을 처음부터 넣어뒀다(소분류 v4의 교훈: 구조적/관찰 가능한 기준만 사용).

In [6]:
if RUN_VLM:
    from vlm_control import query_major_category

    vlm_major_rows = []
    for s in real_samples:
        pred_major, raw = query_major_category(s.image_path)
        truth_major = major_of(s.subcategory)
        vlm_major_rows.append(
            {
                "image_file": s.image_file,
                "truth_major": truth_major,
                "pred_major": pred_major,
                "correct": pred_major == truth_major,
                "raw": raw,
            }
        )
    vlm_major_df = pd.DataFrame(vlm_major_rows)
    vlm_major_acc = vlm_major_df["correct"].mean()
    print(f"[VLM] 대분류 top1 = {vlm_major_acc:.2f}  (CLIP 대비 {clip_major_acc:.2f})")
    display(vlm_major_df)
else:
    print("RUN_VLM=False — 위 §1 셀에서 True로 바꾸고 재실행할 것.")


W0803 13:02:03.138000 14484 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights: 100%|██████████| 713/713 [00:05<00:00, 128.67it/s]


[VLM] 대분류 top1 = 0.94  (CLIP 대비 0.35)


,image_file,truth_major,pred_major,correct,raw
0,denim.jpg,BOTTOM,BOTTOM,True,BOTTOM
1,black_washed.jpg,TOP,TOP,True,TOP
2,fix.jpg,BOTTOM,BOTTOM,True,BOTTOM
3,levis_patch.jpg,BOTTOM,BOTTOM,True,BOTTOM
4,logo_test.jpg,TOP,TOP,True,TOP
5,scratch.jpg,BOTTOM,BOTTOM,True,BOTTOM
6,small_pro.jpg,TOP,TOP,True,TOP
7,navy_logo.png,TOP,TOP,True,TOP
8,vintage_t.png,TOP,TOP,True,TOP
9,black_stain.png,TOP,TOP,True,TOP


## 4. 대분류 비교 요약

CLIP(zero-shot, CPU 가능)과 VLM(Qwen, GPU 필요) 중 대분류 6-way만 놓고 보면 어느 쪽이
나은지, 그리고 "CLIP에게 대분류를 직접 묻기 vs 소분류로 물어서 역산하기" 중 어느 쪽이
나은지를 한 표에서 비교한다.

In [7]:
summary_rows = [
    {"method": f"CLIP 직접 대분류 질의({CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY})", "major_acc": clip_major_acc},
    {
        "method": f"CLIP 소분류 질의 후 역산({CATEGORY_BACKEND}/{CATEGORY_TEMPLATE_KEY})",
        "major_acc": clip_sub_to_major_acc,
    },
]
if RUN_VLM:
    summary_rows.append({"method": "VLM(qwen3-vl-4b-instruct) 직접 대분류 질의", "major_acc": vlm_major_acc})
pd.DataFrame(summary_rows)


,method,major_acc
0,CLIP 직접 대분류 질의(clip_vit_b32/en_clip_style),0.352941
1,CLIP 소분류 질의 후 역산(clip_vit_b32/en_clip_style),1.000000
2,VLM(qwen3-vl-4b-instruct) 직접 대분류 질의,0.941176


## 5. 색상 분류 — 기존 설정 그대로 + 프롬프트/전처리 고도화 재실측

`product_info_check.ipynb`와 같은 정답셋·팔레트를 그대로 쓴다. 지금까지 나온 개선안을
전부 한 자리에서 비교한다:

- CLIP zero-shot(`clip_vit_b32`) — 기준선
- VLM v1(화이트밸런스 기본 적용, `query_color()` 기본값) — `REPORT.md` §13 실측 0.90
- VLM v2_anchored / v3_khaki_gray + 화이트밸런스 — **이 조합은 처음 테스트한다.**
  지난번 v2/v3 실측(§12)은 화이트밸런스 적용 **전**이었다 — 조명 편향 자체가 v2/v3
  회귀의 숨은 원인이었을 수 있어서, 화이트밸런스로 입력을 먼저 정리한 뒤에도 v2/v3가
  여전히 v1보다 나쁜지 다시 확인한다.

In [8]:
from evaluate import evaluate_color

clip_color_preds = evaluate_color(gt.samples, "clip_vit_b32")
clip_color_acc = sum(p.correct_primary for p in clip_color_preds) / len(clip_color_preds)
print(f"[CLIP clip_vit_b32] 색상 primary_acc = {clip_color_acc:.2f}")


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 19122.89it/s]


[CLIP clip_vit_b32] 색상 primary_acc = 0.41


In [9]:
if RUN_VLM:
    from vlm_control import COLOR_PROMPT_VERSIONS, query_color

    color_prompt_wb_rows = []
    for version in COLOR_PROMPT_VERSIONS:
        correct = 0
        for s in real_samples:
            pred_primary, _sec, _raw = query_color(s.image_path, prompt_version=version)
            correct += pred_primary == s.primary_color
        color_prompt_wb_rows.append(
            {
                "prompt_version": version,
                "white_balance": True,
                "n": len(real_samples),
                "primary_acc": correct / len(real_samples),
            }
        )
    color_prompt_wb_df = pd.DataFrame(color_prompt_wb_rows)
    display(color_prompt_wb_df)
else:
    print("RUN_VLM=False — §1 셀에서 True로 바꾸고 재실행할 것.")


,prompt_version,white_balance,n,primary_acc
0,v1,True,17,0.647059
1,v2_anchored,True,17,0.705882
2,v3_khaki_gray,True,17,0.705882


## 6. 색상 분류 — 팔레트 스와치 참조(색상 개선안 4번)

`vlm_control.query_color_with_swatches()` — 상품 사진 아래에 팔레트 hex 스와치 그리드를
합성해 같은 이미지로 같이 보여준다(`color.build_swatch_reference()`). 텍스트로 색 이름을
"기억"해서 매칭하는 대신, 이미지 대 이미지로 비교 판단하게 유도하는 방법 — 여태 프롬프트
문구만 바꾸는 시도(v2/v3)는 다 whack-a-mole이었으니, 아예 다른 축(시각적 참조)을 시도해
본다. 화이트밸런스는 여기서도 기본 적용된다(상품 사진 쪽에만 적용, 스와치는 고정 hex라
보정 대상이 아니다).

In [10]:
if RUN_VLM:
    from vlm_control import query_color_with_swatches

    swatch_rows = []
    for s in real_samples:
        pred_primary, pred_secondary, _raw = query_color_with_swatches(s.image_path)
        swatch_rows.append(
            {
                "image_file": s.image_file,
                "truth_primary": s.primary_color,
                "pred_primary": pred_primary,
                "pred_secondary": pred_secondary,
                "correct": pred_primary == s.primary_color,
            }
        )
    swatch_df = pd.DataFrame(swatch_rows)
    swatch_acc = swatch_df["correct"].mean()
    print(f"[VLM + 스와치 참조] 색상 primary_acc = {swatch_acc:.2f}")
    display(swatch_df)
else:
    print("RUN_VLM=False — §1 셀에서 True로 바꾸고 재실행할 것.")


[VLM + 스와치 참조] 색상 primary_acc = 0.47


,image_file,truth_primary,pred_primary,pred_secondary,correct
0,denim.jpg,BLUE,GRAY,BEIGE,False
1,black_washed.jpg,BLACK,GRAY,NAVY,False
2,fix.jpg,BLUE,BEIGE,NAVY,False
3,levis_patch.jpg,BLUE,GRAY,NAVY,False
4,logo_test.jpg,WHITE,WHITE,NaN,True
5,scratch.jpg,BLUE,BEIGE,NAVY,False
6,small_pro.jpg,WHITE,IVORY,BEIGE,False
7,navy_logo.png,NAVY,NAVY,BLUE,True
8,vintage_t.png,KHAKI,KHAKI,GRAY,True
9,black_stain.png,BLACK,NAVY,NaN,False


## 7. 색상 비교 요약

지금까지 실측한 색상 방법론을 전부 한 표에 모은다 — `REPORT.md` §8~§13에 흩어져 있던
CLIP/VLM/프롬프트/전처리/스와치 결과의 최종 스코어보드.

In [11]:
color_summary_rows = [{"method": "CLIP zero-shot(clip_vit_b32)", "primary_acc": clip_color_acc}]
if RUN_VLM:
    for _, row in color_prompt_wb_df.iterrows():
        color_summary_rows.append(
            {"method": f"VLM {row['prompt_version']} + 화이트밸런스", "primary_acc": row["primary_acc"]}
        )
    color_summary_rows.append({"method": "VLM v1 + 스와치 참조", "primary_acc": swatch_acc})
pd.DataFrame(color_summary_rows)


,method,primary_acc
0,CLIP zero-shot(clip_vit_b32),0.411765
1,VLM v1 + 화이트밸런스,0.647059
2,VLM v2_anchored + 화이트밸런스,0.705882
3,VLM v3_khaki_gray + 화이트밸런스,0.705882
4,VLM v1 + 스와치 참조,0.470588


## 참고

- 소분류(30종) 결과·카테고리 프롬프트 v1~v4·색상 §8~§13 전체 서술은
  `product_info_check.ipynb`와 `REPORT.md` 참고.
- 이 노트북에서 나온 의미 있는 결과는 `REPORT.md`에도 반영해 둘 것 — 출력은 재실행하면
  사라진다.